### Chunking Preprocessed Text for Analysis

In [24]:
#  Load + Chunk Papers ────────────
import json
import os

# Load saved papers
with open('../data/raw/alzheimer_papers.json') as f:
    papers = json.load(f)

print(f" Loaded: {len(papers):,} papers")

def chunk_abstract(paper, 
                   chunk_size=150,
                   overlap=30):
    """
    Split abstract into overlapping chunks
    
    chunk_size = words per chunk
    overlap    = shared words between chunks
    """
    words    = paper['abstract'].split()
    chunks   = []
    start    = 0

    while start < len(words):
        end   = start + chunk_size
        chunk = " ".join(words[start:end])

        chunks.append({
            "chunk_id": f"{paper['pmid']}_{len(chunks)}",
            "pmid":     paper['pmid'],
            "title":    paper['title'],
            "authors":  paper['authors'],
            "year":     paper['year'],
            "journal":  paper['journal'],
            "keywords": paper['keywords'],
            "chunk":    chunk,
            "source":   "PubMed"
        })

        start += chunk_size - overlap
        if start >= len(words):
            break

    return chunks

# ── Chunk all papers ───────────────────────
all_chunks = []

for paper in papers:
    chunks = chunk_abstract(paper)
    all_chunks.extend(chunks)

# ── Save chunks ────────────────────────────
os.makedirs('../data/processed', exist_ok=True)
CHUNKS_FILE = '../data/processed/chunks.json'

with open(CHUNKS_FILE, 'w') as f:
    json.dump(all_chunks, f, indent=2)

# ── Stats ──────────────────────────────────
print(f"\n{'='*50}")
print(f" CHUNKING REPORT")
print(f"{'='*50}")
print(f"   Papers:          {len(papers):,}")
print(f"   Total chunks:    {len(all_chunks):,}")
print(f"   Avg chunks/paper:{len(all_chunks)//len(papers)}")
print(f"   Chunk size:      150 words")
print(f"   Overlap:         30 words")
print(f"   Saved to:        {CHUNKS_FILE}")
print(f"{'='*50}")
print(f" Chunking complete!")


 Loaded: 16,281 papers

 CHUNKING REPORT
   Papers:          16,281
   Total chunks:    40,995
   Avg chunks/paper:2
   Chunk size:      150 words
   Overlap:         30 words
   Saved to:        ../data/processed/chunks.json
 Chunking complete!


### Generate Embeddings 

In [ ]:
### Generate Embeddings ────────────

from sentence_transformers import SentenceTransformer
import numpy as np
import os

print(" Loading PubMedBERT model...")
model = SentenceTransformer(
    'neuml/pubmedbert-base-embeddings'
)
print(" Model loaded!")

# Extract text only
texts = [c['chunk'] for c in all_chunks]

print(f"\n⏳ Embedding {len(texts):,} chunks...")
print(f"   Model:      PubMedBERT 768-dim")
print(f"   Chunks:     {len(texts):,}")
print(f"   Time:       ~30-45 minutes ⏳")
print(f"   Go get coffee! ☕")

# Generate embeddings
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings complete!")
print(f"   Shape: {embeddings.shape}")
print(f"   Expected: ({len(texts):,}, 768)")


 Loading PubMedBERT model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 794.71it/s, Materializing param=pooler.dense.weight]                               


✅ Model loaded!

⏳ Embedding 40,995 chunks...
   Model:      PubMedBERT 768-dim
   Chunks:     40,995
   Time:       ~30-45 minutes ⏳
   Go get coffee! ☕


Batches:   1%|▏         | 9/641 [03:21<3:55:20, 22.34s/it]


KeyboardInterrupt: 

In [22]:
#  Save Embeddings ────────────────
import numpy as np
import os

os.makedirs('../data/processed', exist_ok=True)
EMBEDDINGS_FILE = '../data/processed/embeddings.npy'

np.save(EMBEDDINGS_FILE, embeddings)

size_mb = os.path.getsize(EMBEDDINGS_FILE)/1024/1024

print(f" Embeddings saved!")
print(f"   File:  {EMBEDDINGS_FILE}")
print(f"   Shape: {embeddings.shape}")
print(f"   Size:  {size_mb:.1f} MB")


 Embeddings saved!
   File:  ../data/processed/embeddings.npy
   Shape: (40995, 768)
   Size:  120.1 MB


### Final Report

In [20]:
# ── CELL 8: Final Report ───────────────────
print(f"\n{'='*50}")
print(f" PREPROCESSING COMPLETE!")
print(f"{'='*50}")
print(f"\n SUMMARY:")
print(f"   Papers:      {len(papers):,}")
print(f"   Chunks:      {len(all_chunks):,}")
print(f"   Embeddings:  {embeddings.shape}")
print(f"   Dimensions:  768 (PubMedBERT)")
print(f"   Model:       neuml/pubmedbert-base-embeddings")
print(f"\n FILES SAVED:")
print(f"   ../data/processed/chunks.json")
print(f"    ../data/processed/embeddings.npy")
print(f"\n NEXT STEP:")
print(f"   → Build FAISS index")
print(f"   → vector_store/faiss_index/")
print(f"{'='*50}")


 PREPROCESSING COMPLETE!

 SUMMARY:
   Papers:      16,281
   Chunks:      40,995
   Embeddings:  (40995, 768)
   Dimensions:  768 (PubMedBERT)
   Model:       neuml/pubmedbert-base-embeddings

 FILES SAVED:
   ../data/processed/chunks.json
    ../data/processed/embeddings.npy

 NEXT STEP:
   → Build FAISS index
   → vector_store/faiss_index/
